In [ ]:
pip install langchain_community langchain_text_splitters langsmith bs4 tiktoken langchain-openai langchainhub chromadb langchain youtube-transcript-api pytube langchain-google-genai

In [68]:
import os
from dotenv import load_dotenv

load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
# os.environ['LANGCHAIN_ENDPOINT'] = ''
# os.environ['LANGCHAIN_API_KEY'] = ''
# os.environ['LANGSMITH_API_KEY'] = ''
# os.environ['GOOGLE_API_KEY'] = ''


In [69]:

import bs4
from langsmith import Client
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings

##Indexing##

In [70]:
#load documents
loader = WebBaseLoader (
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only = bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

#split text into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
#embed
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

retriever = vectorstore.as_retriever()

## Retrieval and Generation ##
langsmith_api = os.environ['LANGSMITH_API_KEY']
client =  Client(api_key=langsmith_api)
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

#LLM 
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature = 0,
    max_tokens=None,
    max_retries=2,
)

#post-processing
def format_docs(docs):
    docs_as_text = "\n\n".join(doc.page_content for doc in docs)
    print("retrived information from docs:", docs_as_text)
    return docs_as_text
#Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    |llm
    |StrOutputParser()
)


In [71]:
#Questin
rag_chain.invoke("What is task decomposition?")

retrived information from docs: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner which is common in certain robotic setups but not in many other domain

'Task decomposition is the process of breaking down a task into smaller steps or subgoals. This can be achieved by using Large Language Models (LLMs) with simple prompts or task-specific instructions. Human input can also be utilized for task decomposition.'

In [ ]:
#Focus on Indexing

In [50]:
question = "What kind of food do i like ?"
document = "My favourite food is chicken rice."

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector1 = embedding.embed_query(question)
vector2 = embedding.embed_query(document)
# print(vector, vector2)

[-0.011445953510701656, -0.002427895786240697, 0.021003900095820427, -0.07024472951889038, 0.00389140285551548, 0.008805411867797375, -0.0038491522427648306, 0.0033590677194297314, 0.012469986453652382, 0.0013473392464220524, -0.005604222416877747, -0.009712664410471916, 0.0017451931489631534, -0.0008251264225691557, 0.1455010324716568, -0.022381331771612167, -0.017387526109814644, 0.001186365494504571, 0.01051147561520338, -0.014213423244655132, 0.010739631950855255, 0.004497390706092119, 0.01472817175090313, -0.00918087549507618, -0.0010048608528450131, 0.012858291156589985, 0.0023464965634047985, 0.00459393672645092, 0.010730917565524578, -4.494192035053857e-05, 0.005144511349499226, 0.01881985366344452, 0.017676040530204773, 0.03559799864888191, 0.012839391827583313, 0.01718149706721306, 0.0013081827200949192, -0.007212865632027388, 0.012786377221345901, -0.013949228450655937, -0.013927173800766468, -0.001257249852642417, -0.00988216046243906, 0.006910242605954409, 0.02244300022721

##Retrieval

In [66]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma

gemini_embedding_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorStore = Chroma.from_documents(documents=splits, embedding=gemini_embedding_model)

retriever = vectorStore.as_retriever(search_kwargs={"k":1})
docs = retriever.invoke("What is Task Decomposition?")

(docs)

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.\nAnother quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner